# Bronze Orchestrator: price_audit
Orchestrates ingestion, validation, and monitoring for the Bronze layer of price_audit.

**Execution Order:**
1. Ingestion
2. Validation
3. Monitoring

**Alerts:**
- Execution errors are captured and displayed for each step.
- If any step fails, subsequent steps are not executed.

In [ ]:
# Import orchestrated functions
import sys
sys.path.append("/Workspace/Users/diego.mayorgacapera@gmail.com/.bundle/BI_Market_Visibility/dev/files")
from src.bronze.price_audit.ingest_price_audit import run_ingestion
from src.bronze.price_audit.validate_price_audit import run_validation
from src.bronze.price_audit.monitor_price_audit import run_monitoring

In [ ]:
# --- Logging setup (must be first for consistent context)
import logging
logger = logging.getLogger('bronze_orchestrator')
if not logger.hasHandlers():
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)

In [ ]:
# Force reload of updated modules to ensure notebook uses latest code from bundle
import importlib
import src.bronze.price_audit.ingest_price_audit as ingest_price_audit_mod
importlib.reload(ingest_price_audit_mod)
run_ingestion = ingest_price_audit_mod.run_ingestion

In [ ]:
# --- Environment Parameters ---
env = 'dev'
bronze_table = 'workspace.bronze.price_audit'
validation_table = 'workspace.bronze.price_audit_validation'

In [ ]:
# Step 1: Ingestion
try:
    batch_id, rows_ingested = run_ingestion(source_path='/Volumes/workspace/raw_data/price_audit', delta_table=bronze_table, env=env)
    print(f'✅ Ingestion completed. Batch ID: {batch_id}, Rows: {rows_ingested}')
except Exception as e:
    print(f'❌ Ingestion failed: {str(e)}')
    raise

In [ ]:
# Step 2: Validation (only if ingestion succeeded and rows_ingested > 0)
if rows_ingested == 0 or batch_id is None:
    print('⚠️ No new data ingested. Skipping validation.')
    metrics_df = None
else:
    try:
        metrics_df = run_validation(bronze_table=bronze_table, env=env)
        print('✅ Validation completed.')
        display(metrics_df)
    except Exception as e:
        print(f'❌ Validation failed: {str(e)}')
        raise

In [ ]:
# Step 3: Monitoring (only if validation succeeded and rows_ingested > 0)
if rows_ingested == 0 or batch_id is None:
    print('⚠️ No new data ingested. Skipping monitoring.')
    metrics_json, alerts_json = [], []
else:
    try:
        metrics_json, alerts_json = run_monitoring(bronze_table=bronze_table, validation_table=validation_table, env=env)
        print('✅ Monitoring completed.')
        print('Metrics:')
        print(metrics_json)
        print(f'Metrics volume: {len(metrics_json)}')
        if alerts_json:
            print('⚠️ Alerts:')
            print(alerts_json)
            print(f'Alerts volume: {len(alerts_json)}')
        else:
            print('No alerts triggered.')
    except Exception as e:
        print(f'❌ Monitoring failed: {str(e)}')
        raise